In [104]:
# Segmentation Pipeline
# Input: 
#   Dicom Files - list of dicom files in one folder
# Output:
#   Dicom Files (Bias correction, skull stripping and segmentation) - list of dicom files in one folder
#       Segmented regions:  White Matter, Gray Matter, CSF, Background
# 
# Main function is located before visualization cells. It loops through all dicom folders (all patients),
# and saves a cooresponding dicom folder named "AnatomicalSeg" located near the original files. Each
# dicom is matched with an original dicom (090.dcm will be the segmented dicom of the original 090.dcm)

# Import statements
import pydicom
import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
from dipy.segment.tissue import TissueClassifierHMRF
from dipy.io.image import load_nifti
from matplotlib.colors import LinearSegmentedColormap
import nibabel as nib
from nibabel import Nifti1Image 
from ipywidgets import interact
import SimpleITK as sitk
from dipy.segment.mask import median_otsu
from dipy.core.histeq import histeq
from nilearn import image
from deepbrain import Extractor
import numpy.ma as ma
import dicom2nifti
import time

# Define the values and colors for segmentation cmap
colors = ['black', 'purple', 'orange', 'firebrick']
values = [0, 1, 2, 3]
labels = ['CSF', 'GM', 'WM']

# Create a custom colormap dictionary
seg_cmap = LinearSegmentedColormap.from_list("mycmap", list(zip(values, colors)))

In [131]:
def biasCorrection(nifti, fileName):

    # print("Currently Bias Correcting:", fileName)

    # Transform nifti to Array
    raw_img_sitk = sitk.GetImageFromArray(nifti.astype(np.float32))

    # Rescale and apply Threshold of correction
    transformed = sitk.RescaleIntensity(raw_img_sitk, 0, 255)
    transformed = sitk.LiThreshold(transformed,0,1)
    head_mask = transformed

    # Shrink image (may need to adjust this setting)
    shrinkFactor = 4
    inputImage = raw_img_sitk
    inputImage = sitk.Shrink( raw_img_sitk, [ shrinkFactor ] * inputImage.GetDimension() )
    maskImage = sitk.Shrink( head_mask, [ shrinkFactor ] * inputImage.GetDimension() ) 

    # Obtain correcter
    bias_corrector = sitk.N4BiasFieldCorrectionImageFilter()

    # Execute with Image and Mask, Get Bias Field
    bias_corrector.Execute(inputImage, maskImage)
    log_bias_field = bias_corrector.GetLogBiasFieldAsImage(raw_img_sitk)

    # Compute Corrected Image
    bias_corrected_nifti = raw_img_sitk / sitk.Exp( log_bias_field)

    # Get full-resolution image as Numpy array (from sitk image)
    corrected_image_full_resolution_array = sitk.GetArrayFromImage(bias_corrected_nifti)
    corrected_image_full_resolution_nifti = nib.Nifti1Image(corrected_image_full_resolution_array, affine=None)    

    return corrected_image_full_resolution_nifti

biasCorrection reference link:
https://simpleitk.readthedocs.io/en/master/link_N4BiasFieldCorrection_docs.html

In [132]:
def skullStrip(image, fileName):

    # print("Currently Skull Stripping:", fileName)

    # print("Skull Stripping File: ", fileName)
    ext = Extractor()

    # `prob` will be a 3d numpy image containing probability of being brain tissue for each of the voxels in `img`
    prob = ext.run(image) 

    # Binary Mask
    mask = prob > 0.1 

    # Overlay mask onto original image
    stripped_image = image*mask

    return stripped_image

skullStrip reference link:
https://pypi.org/project/deepbrain/

In [133]:
def Segmentation(image, affine, fileName):

    print("Currently Segmenting:", fileName)
    
    # Num of classes and beta value
    nclass = 3
    beta = 0.1

    # Compute segmentation
    hmrf = TissueClassifierHMRF()
    # print("Segmenting File: ", fileName)
    _, final_segmentation, _ = hmrf.classify(image, nclass, beta)

    segmentation = Nifti1Image(final_segmentation, affine)

    return segmentation

Segmentation reference link:
https://dipy.org/documentation/1.0.0./examples_built/tissue_classification/

In [108]:
def writeSlices(series_tag_values, new_img, i, out_dir):
    image_slice = new_img[:, :, i]
    writer = sitk.ImageFileWriter()
    writer.KeepOriginalImageUIDOn()

    # Tags shared by the series.
    list(map(lambda tag_value: image_slice.SetMetaData(tag_value[0], tag_value[1]), series_tag_values))

    # Slice specific tags.
    image_slice.SetMetaData("0008|0012", time.strftime("%Y%m%d"))  # Instance Creation Date
    image_slice.SetMetaData("0008|0013", time.strftime("%H%M%S"))  # Instance Creation Time

    # Setting the type to CT preserves the slice location.
    image_slice.SetMetaData("0008|0060", "CT")  # set the type to CT so the thickness is carried over

    # (0020, 0032) image position patient determines the 3D spacing between slices.
    image_slice.SetMetaData("0020|0032", '\\'.join(map(str, new_img.TransformIndexToPhysicalPoint((0, 0, i)))))  # Image Position (Patient)
    image_slice.SetMetaData("0020,0013", str(i))  # Instance Number

    # Write to the output directory and add the extension dcm, to force writing in DICOM format.
    writer.SetFileName(os.path.join(out_dir, str(i).zfill(3) + '.dcm'))
    writer.Execute(image_slice)




def nifti2dicom_1file(in_dir):
    """
    This function is to convert only one nifti file into dicom series

    `nifti_dir`: the path to the one nifti file
    `out_dir`: the path to output
    """

    # Access segmented nifti file from input directory
    out_dir = os.path.join(os.path.dirname(in_dir), "anatomicalSeg")
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    new_img = sitk.ReadImage(in_dir) 
    modification_time = time.strftime("%H%M%S")
    modification_date = time.strftime("%Y%m%d")

    direction = new_img.GetDirection()
    series_tag_values = [("0008|0031",modification_time), # Series Time
                    ("0008|0021",modification_date), # Series Date
                    ("0008|0008","DERIVED\\SECONDARY"), # Image Type
                    ("0020|000e", "1.2.826.0.1.3680043.2.1125."+modification_date+".1"+modification_time), # Series Instance UID
                    ("0020|0037", '\\'.join(map(str, (direction[0], direction[3], direction[6],# Image Orientation (Patient)
                                                        direction[1],direction[4],direction[7])))),
                    ("0008|103e", "Created-Pycad")] # Series Description

    # Write slices to output directory
    list(map(lambda i: writeSlices(series_tag_values, new_img, i, out_dir), range(new_img.GetDepth())))


writeSlices reference link:
https://simpleitk.readthedocs.io/en/master/link_DicomSeriesFromArray_docs.html

In [109]:
def save_segmentation_nifti(in_dir, seg_nifti):

    # Create output directory in same location as input directory
    out_dir = os.path.join(os.path.dirname(in_dir), "NIFTI_Seg_temp.nii")

    # Save nifti to output directory
    nib.save(seg_nifti,out_dir)

    # Return path of saved nifti
    return out_dir

In [111]:
def pipeline(input_dir):

    # Convert Dicom to Nifti
    Current_Nifti = dicom2nifti.convert_dicom.dicom_series_to_nifti(input_dir, output_file=None, reorient_nifti=False)
    nifti = Current_Nifti["NII"] #Current_Nifti is a dictionary, "NII" is nifti
    nifti_array = nifti.get_fdata()
    affine = nifti.affine# get affine information from nifti file

    # Bias Field Correct
    corrected = biasCorrection(nifti_array, input_dir)
    bias_corrected = corrected.get_fdata()

    # Skull Strip
    stripped = skullStrip(bias_corrected, input_dir)
    
    # Segmentation
    segmented = Segmentation(stripped, affine, input_dir)

    # Save Segmentation as NIFTI (otherwise conversion won't work)
    saved_dir = save_segmentation_nifti(input_dir, segmented)

    # Convert NIFTI to Dicom
    nifti2dicom_1file(saved_dir)
    
    return bias_corrected, stripped, segmented # remove

In [1]:
### MAIN FUNCTION
# Input Directory: "E:\\Documents\\MTLE\\LP-xxxx\\LP-xxxx-xx-xx-xx\\anatomicalData"
# Output Directory: "E:\\Documents\\MTLE\\LP-xxxx\\LP-xxxx-xx-xx-xx\\anatomicalSeg"

root_dir = "E:\\Documents\\MTLE"
target_folder = "anatomicalData"

for dirpath, dirnames, filenames in os.walk(root_dir):
    if "anatomicalData" in dirnames:
        target_folder_path = os.path.join(dirpath, "anatomicalData")
        print("Current anatomicalData Folder:", target_folder_path)

        # Bias Correct, Skull Strip, Segment
        bias, strip, seg = pipeline(target_folder_path)

